In [21]:
import pandas as pd 
import json
import re
import networkx as nx
from itertools import combinations as comb
from collections import Counter

In [22]:
df = pd.read_csv("/Users/louiseppel/Documents/UMASS/data_vis/louis_assignment_2/data_scopus.csv")
print("Before dropping:", df.shape[0])
df = df.dropna(subset=['Year', 'Authors', 'Authors with affiliations'])
print("After dropping:", df.shape[0])


Before dropping: 570
After dropping: 561


In [23]:
def parse_author_affiliation_entry(entry):
    # Scopus format is "Surname, Initials., Affiliation text..." - first two comma segments are the name
    parts = [p.strip() for p in entry.split(',')]
    name = ", ".join(parts[:2])
    affiliation = ", ".join(parts[2:])
    return name, affiliation

def split_authors_with_affiliations(text):
    entries = [e.strip() for e in re.split(r';\s*', str(text)) if e.strip()]
    return [parse_author_affiliation_entry(e) for e in entries]

df['parsed_affiliations'] = df['Authors with affiliations'].apply(split_authors_with_affiliations)


In [36]:
COUNTRY_ALIASES = {
    "Russian Federation": "Russia",
    "United Kingdom": "UK",
    "USA": "United States",
    "U.S.A": "United States",
    "Korea": "South Korea",
    "Hong Kong": "China",
}

def extract_country(affiliation):
    # the country is reliably the final comma-separated segment of a Scopus affiliation string
    if not affiliation:
        return None
    segment = affiliation.split(',')[-1].strip().rstrip('.')
    if not segment:
        return None
    return COUNTRY_ALIASES.get(segment, segment)

def build_paper_authors(row):
    ids = [i.strip() for i in str(row['Author(s) ID']).split(';') if i.strip()]
    entries = row['parsed_affiliations']
    authors = []
    seen = set()
    for author_id, (name, affiliation) in zip(ids, entries):
        country = extract_country(affiliation)
        if author_id in seen:
            continue
        seen.add(author_id)
        if country is not None:
            authors.append({"id": author_id, "name": name, "country": country})
    return authors

df['paper_authors'] = df.apply(build_paper_authors, axis=1)


In [37]:
df['paper_authors'][20]


[{'id': '57196245087', 'name': 'Komissarova, T.', 'country': 'Russia'},
 {'id': '57196235017', 'name': 'Gadzhieva, E.', 'country': 'Russia'},
 {'id': '8901086000', 'name': 'Lebedeva, M.', 'country': 'Russia'},
 {'id': '57196243666', 'name': 'Morozova, O.', 'country': 'Russia'}]

In [38]:
df.head()

,Title,Year,EID,Abstract,Publisher,Conference name,Conference date,Authors,Author(s) ID,Authors with affiliations,Source title,Abbreviated Source Title,Cited by,parsed_affiliations,paper_authors
0,Virtual reality applications for the built env...,2020,2-s2.0-85086464158,With its advanced capabilities of immersive an...,Elsevier B.V.,NaN,NaN,"Zhang Y., Liu H., Kang S.-C., Al-Hussein M.",57202536919;53984550800;57215426514;6603541102;,"Zhang, Y., Department of Civil and Environment...",Automation in Construction,Autom Constr,NaN,"[(Zhang, Y., Department of Civil and Environme...","[{'id': '57202536919', 'name': 'Zhang, Y.', 'c..."
1,"Self-tracking while doing sport: Comfort, moti...",2020,2-s2.0-85082875828,The spread of wearable technologies is paving ...,Academic Press,NaN,NaN,"Rapp A., Tirabeni L.",23398572100;57191836654;,"Rapp, A., Computer Science Department, Univers...",International Journal of Human Computer Studies,Int J Hum Comput Stud,NaN,"[(Rapp, A., Computer Science Department, Unive...","[{'id': '23398572100', 'name': 'Rapp, A.', 'co..."
2,"Bridge damage: Detection, IFC-based semantic e...",2020,2-s2.0-85078194587,Building Information Modeling (BIM) representa...,Elsevier B.V.,NaN,NaN,"Isailović D., Stojanovic V., Trapp M., Richter...",57205293410;56421700900;24831175200;3619515940...,"Isailović, D., Department for Construction Pro...",Automation in Construction,Autom Constr,NaN,"[(Isailović, D., Department for Construction P...","[{'id': '57205293410', 'name': 'Isailović, D.'..."
3,VR system for spatio-temporal visualization of...,2019,2-s2.0-85075706132,Social media analysis is helpful to understand...,Springer,NaN,NaN,"Okada K., Yoshida M., Itoh T., Czauderna T., S...",57201502480;57171240600;35373203100;2592569550...,"Okada, K., Ochanomizu University, Tokyo, Japan...",Multimedia Tools and Applications,Multimedia Tools Appl,NaN,"[(Okada, K., Ochanomizu University, Tokyo, Jap...","[{'id': '57201502480', 'name': 'Okada, K.', 'c..."
4,DiseaSE: A biomedical text analytics system fo...,2019,2-s2.0-85074886243,Due to increasing volume and unstructured natu...,Academic Press Inc.,NaN,NaN,"Abulaish M., Parwez M.A., Jahiruddin",6505934038;57202719622;35590173900;,"Abulaish, M., Department of Computer Science, ...",Journal of Biomedical Informatics,J. Biomed. Informatics,NaN,"[(Abulaish, M., Department of Computer Science...","[{'id': '6505934038', 'name': 'Abulaish, M.', ..."


In [39]:
G = nx.Graph()
country_votes = {}  # author_id -> Counter(country) across all their papers

for authors in df['paper_authors']:
    for a in authors:
        if a['id'] not in G:
            G.add_node(a['id'], name=a['name'])
        country_votes.setdefault(a['id'], Counter())[a['country']] += 1

    ids = [a['id'] for a in authors]
    for u, v in comb(sorted(set(ids)), 2):
        if G.has_edge(u, v):
            G[u][v]['weight'] += 1
        else:
            G.add_edge(u, v, weight=1)

print("Nodes:", G.number_of_nodes(), "Edges:", G.number_of_edges())


Nodes: 1536 Edges: 3026


In [40]:
# resolve each author's country as the mode across all their papers (affiliations can vary by paper)
for node_id in G.nodes:
    G.nodes[node_id]['country'] = country_votes[node_id].most_common(1)[0][0]
    G.nodes[node_id]['degree'] = G.degree(node_id)


In [48]:
PALETTE = ['#a6cee3','#1f78b4','#b2df8a','#33a02c','#fb9a99','#e31a1c','#fdbf6f','#ff7f00','#cab2d6','#6a3d9a']
OTHER_COLOR = "#A9A9A9"

country_counts = Counter(nx.get_node_attributes(G, 'country').values())
top10_countries = [c for c, _ in country_counts.most_common(10)]
country_color = dict(zip(top10_countries, PALETTE))

for node_id in G.nodes:
    G.nodes[node_id]['color'] = country_color.get(G.nodes[node_id]['country'], OTHER_COLOR)

print("Top 10 countries:", top10_countries)


Top 10 countries: ['United States', 'Germany', 'China', 'UK', 'South Korea', 'Japan', 'Brazil', 'Russia', 'Canada', 'Spain']


In [49]:
from networkx.readwrite import json_graph

try:
    graph_data = json_graph.node_link_data(G, edges="links")
except TypeError:
    graph_data = json_graph.node_link_data(G)

output = {"nodes": graph_data["nodes"], "links": graph_data["links"]}

with open("data.json", "w") as f:
    json.dump(output, f, indent=2)

print("Wrote data.json:", len(output['nodes']), "nodes,", len(output['links']), "links")


Wrote data.json: 1536 nodes, 3026 links


In [45]:
nx.write_gexf(G, "authors.gexf")